# Session 6 — Multi-Head Latent Attention (DeepSeek-V2 style)

Caches a low-rank latent per token instead of per-head K/V.

In [1]:
import sys; sys.path.insert(0, '../src')
import torch
from kv_cache_variants.attention.mla import MultiHeadLatentAttention
from kv_cache_variants.rope import build_rope_cache

torch.manual_seed(0)
d_model, num_heads, latent_dim, rope_dim = 32, 4, 8, 8
mla = MultiHeadLatentAttention(d_model, num_heads, latent_dim, rope_dim)
cos, sin = build_rope_cache(rope_dim, max_seq_len=16)
x = torch.randn(1, 5, d_model)
out, (latent, k_rope) = mla(x, rope=(cos, sin))
print('output:', out.shape, '| cached latent:', latent.shape, '| cached k_rope:', k_rope.shape)

output: torch.Size([1, 5, 32]) | cached latent: torch.Size([1, 5, 8]) | cached k_rope: torch.Size([1, 5, 8])


In [2]:
head_dim = d_model // num_heads
cached_per_token = latent.shape[-1] + k_rope.shape[-1]
full_kv_per_token = 2 * num_heads * head_dim
print(f'MLA cached elems/token: {cached_per_token} vs full K/V elems/token: {full_kv_per_token}')

MLA cached elems/token: 16 vs full K/V elems/token: 64
